In [4]:
import pandas as pd
import numpy as np

# Charger
df = pd.read_excel("/Users/thibaultdautheville/Downloads/REI département total.xlsx")






In [5]:
import sys
!{sys.executable} -m pip install scikit-learn


In [10]:
import sklearn.linear_model


In [6]:
import pandas as pd
import numpy as np



In [7]:


from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler



In [8]:
# On récupère toutes les colonnes qui commencent par "FB -"
# Ce sont nos variables explicatives potentielles
colonnes_fb = [col for col in df.columns if col.startswith('FB -')]

print(f"{len(colonnes_fb)} colonnes FB trouvées")
print(colonnes_fb[:5])  # Aperçu des 5 premières


141 colonnes FB trouvées
["FB - FRAIS D'ASSIETTE, DEGREVEMENT,  NON VALEURS", 'FB - COMMUNE / BASE NETTE', 'FB - COMMUNE / TAUX NET', 'FB - COMMUNE / TAUX VOTÉ', 'FB - COMMUNE / MONTANT RÉEL']


In [48]:
df_dep = df.groupby("DÉPARTEMENT")
print(df_dep.size())  # Affiche le nombre de lignes par département
print(df_dep.index[:10].tolist())


DÉPARTEMENT
1      392
2      798
3      317
4      198
5      162
      ... 
976     17
977      1
978      1
2A     124
2B     236
Length: 103, dtype: int64


AttributeError: 'DataFrameGroupBy' object has no attribute 'index'

In [49]:
df_dep = df.groupby("DÉPARTEMENT")[fb_cols].sum()
print(df_dep.shape)
print(df_dep.index[:5].tolist())


NameError: name 'fb_cols' is not defined

In [9]:
# On somme toutes les colonnes FB par département
# Chaque ligne du df correspond à une commune, 
# on regroupe pour avoir une ligne par département
df_dep = df.groupby('DÉPARTEMENT')[colonnes_fb].sum()

print(f"Shape : {df_dep.shape}")
# On doit avoir ~100 départements x 141 colonnes
print(df_dep.head(3))


Shape : (103, 141)
             FB - FRAIS D'ASSIETTE, DEGREVEMENT,  NON VALEURS  \
DÉPARTEMENT                                                     
1                                                  13954817.0   
2                                                  12487021.0   
3                                                   9750159.0   

             FB - COMMUNE / BASE NETTE  FB - COMMUNE / TAUX NET  \
DÉPARTEMENT                                                       
1                          976264084.0              10828.23647   
2                          558649627.0              35909.30247   
3                          462355072.0              11142.36420   

             FB - COMMUNE / TAUX VOTÉ  FB - COMMUNE / MONTANT RÉEL  \
DÉPARTEMENT                                                          
1                            10828.41                  291806427.0   
2                            35909.50                  288853430.0   
3                            11142.89  

In [38]:
cibles = {
    "01": 88307, "07": 48610, "73": 125779, "74": 106718,
    "75": 750648, "14": 107390, "23": 13378, "17": 73052,
    "64": 80070, "06": 217386, "972": 52999,
    "974": 71438, "84": 89228, "82": 36112,
    "86": 48609, "27": 80602, "78": 251590, "88": 55057,
    "2B": 18164, "56": 85428, "89": 51620,
}


In [ ]:
df_dep = df_dep.loc[df_dep.index.isin(cibles.keys())] #garde uniquement les départements pour lesquels tu as une cible connue (ex: AIN, CALVADOS…). Filtre les lignes.
y = pd.Series(cibles).loc[df_dep.index] #crée le vecteur cible y (les valeurs TFPB connues), aligné dans le même ordre que X.
X = df_dep.fillna(0) #crée la matrice de features X (colonnes FB), en remplaçant les valeurs manquantes par 0.



#En résumé : X = variables Facebook par département, y = recettes TFPB par département. Le Lasso va chercher quelles colonnes FB prédisent le mieux y.

Index([   1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
       ...
         95,  971,  972,  973,  974,  976,  977,  978, '2A', '2B'],
      dtype='object', name='DÉPARTEMENT', length=103)
object


In [28]:

print(df.columns.tolist())


['DÉPARTEMENT', 'DIRECTION', 'CODE COMMUNE', 'COMMUNE RECENSÉE (R si recensée)', 'LIBELLÉ COMMUNE', 'Numéro national du groupement', "Numéro SIREN de l'EPCI", 'Libellé du Groupement', "Option fiscale de l'EPCI (FPA, FPZ ou FPU)", 'Forme juridique EPCI (CC, CA, CU ou Mét)', 'LIBELLÉ DÉPARTEMENT', 'LIBELLÉ RÉGION', "FNB - FRAIS D'ASSIETTE, DEGREVEMENT, NON VALEURS", 'FNB - COMMUNE / BASE NETTE', 'FNB - COMMUNE / TAUX NET', 'FNB - COMMUNE / TAUX VOTÉ', 'FNB - COMMUNE / MONTANT RÉEL', "FNB - COMMUNE / NOMBRE D'ARTICLES", 'FNB - SYNDICATS ET ORG. ASSIMILÉS / BASE NETTE', 'FNB - SYNDICATS ET ORG. ASSIMILÉS / TAUX NET', 'FNB - SYNDICATS ET ORG. ASSIMILÉS / MONTANT RÉEL', 'FNB - GFP / BASE NETTE', 'FNB - GFP / TAUX NET', 'FNB - GFP / TAUX VOTÉ', 'FNB - GFP / MONTANT RÉEL', 'TAFNB - BASE TAXABLE COMMUNALE', 'TAFNB - COMMUNE / TAUX NET', 'TAFNB - COMMUNE / MONTANT RÉEL', 'TAFNB - BASE TAXABLE GFP', 'TAFNB - GFP / TAUX NET', 'TAFNB - GFP / MONTANT RÉEL', 'TAFNB - BASE TAXABLE MÉTROPOLE DU GRAND P

In [32]:
print(df[['DÉPARTEMENT']].head())


  DÉPARTEMENT
0           1
1           1
2           1
3           1
4           1


In [45]:
print(df_dep.shape)
print(df_dep.head())


(1, 141)
             FB - FRAIS D'ASSIETTE, DEGREVEMENT,  NON VALEURS  \
DÉPARTEMENT                                                     
2B                                                  5636640.0   

             FB - COMMUNE / BASE NETTE  FB - COMMUNE / TAUX NET  \
DÉPARTEMENT                                                       
2B                         244731925.0                  5526.48   

             FB - COMMUNE / TAUX VOTÉ  FB - COMMUNE / MONTANT RÉEL  \
DÉPARTEMENT                                                          
2B                            5526.48                   67140537.0   

             FB - COMMUNE / NOMBRE D'ARTICLES  \
DÉPARTEMENT                                     
2B                                   109503.0   

             FB - COMMUNE / LISSAGE - MONTANT  \
DÉPARTEMENT                                     
2B                                    29482.0   

             FB - COMMUNE / LISSAGE - NOMBRE  \
DÉPARTEMENT                           

In [42]:
print(X.shape)
print(df_dep.index.tolist())


(1, 141)
['2B']


In [44]:
# Reconstruire l'index correctement
df_dep.index = df_dep.index.map(lambda x: str(x).zfill(2) if isinstance(x, int) else x)
print(set(df_dep.index) & set(cibles.keys()))
print(df_dep.index[:10].tolist())


{'2B'}
['2B']


In [43]:
print(set(df_dep.index) & set(cibles.keys()))
print(df_dep.index[:5].tolist())
print(list(cibles.keys())[:5])


{'2B'}
['2B']
['01', '07', '73', '74', '75']


In [41]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lasso = LassoCV(cv=5, max_iter=10000).fit(X_scaled, y)


ValueError: Cannot have number of splits n_splits=5 greater than the number of samples: n_samples=1.